In [1]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,path,datapointID,subtaskID,answer
0,./recordings/1.wav,1,1,7
1,./recordings/2.wav,2,1,6
2,./recordings/3.wav,3,1,7
3,./recordings/4.wav,4,1,7
4,./recordings/5.wav,5,1,7


In [2]:
import librosa

def get_feats(path):
    y, sr = librosa.load(path)
    mel = librosa.feature.melspectrogram( y=y, sr=sr, n_mels=64, fmax=350)  # low pass gen doar sub 350
    mel = librosa.power_to_db(mel)
    return list(mel.mean(axis=1)) + list(mel.std(axis=1))

len(get_feats(df_train.path[0]))

128

In [3]:
from tqdm import tqdm

X = []
y = df_train.answer

for path in tqdm(df_train.path):
    feats = get_feats(path)
    X.append(feats)

X_final = []
for path in tqdm(df_test.path):
    feats = get_feats(path)
    X_final.append(feats)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_final = scaler.transform(X_final)

X = pd.DataFrame(X)
X_final = pd.DataFrame(X_final)
X.head()

100%|███████████████████████████████████████████████████████████████████████████████| 200/200 [00:00<00:00, 203.61it/s]


,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
0,0.358972,-0.397422,0.431870,-0.394209,0.484253,-0.374535,0.395350,-0.470457,0.292183,-0.581824,...,-1.286186,-1.280514,-0.313126,-0.314430,1.127251,1.094600,0.679117,0.679116,0.971347,0.971347
1,0.822522,0.901264,-0.424792,-0.229902,-0.517858,-0.348463,-0.645974,-0.432658,-0.751762,-0.478261,...,-0.415448,-0.422492,3.320817,3.287762,-1.045324,-1.026744,0.615948,0.615948,0.056509,0.056509
2,1.781098,1.684614,1.115310,1.030570,0.871290,0.803537,0.707326,0.853297,0.635174,0.736601,...,1.323550,1.291113,1.183085,1.168708,-1.381719,-1.355207,-0.249465,-0.249465,0.645419,0.645418
3,1.264463,0.461587,1.900412,0.796398,1.698410,0.664856,1.675609,0.617068,1.584581,0.528218,...,-0.633217,-0.637081,-0.688039,-0.686068,-1.185910,-1.164015,0.192579,0.192579,1.444933,1.444933
4,-0.718247,-1.070964,-0.596146,-0.783319,-0.724212,-0.609096,-0.686853,-0.314386,-0.407960,-0.045663,...,0.533396,0.512496,1.790300,1.770617,-0.772237,-0.760097,-0.870912,-0.870912,-0.594248,-0.594248


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

model = LogisticRegression()

cross_val_score(model, X, y, cv=10, scoring='accuracy').mean()

np.float64(0.93)

In [9]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [10]:
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.datapointID,
    "answer": y_preds
})

submission

,subtaskID,datapointID,answer
0,1,1,6
1,1,2,7
2,1,3,6
3,1,4,6
4,1,5,7
...,...,...,...
195,1,196,6
196,1,197,6
197,1,198,6
198,1,199,7


In [11]:
submission.to_csv("submission.csv", index=False)